In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
#1. Load data
df = pd.read_csv("./raw_car_dataset.csv")

print("\n--------HEAD DATA--------\n")
print(df.head(10))
print("\n\n--------INFO DATA--------\n")
print(df.info())
print("\n\n--------MISSING--------\n")
print(df.isnull().sum())
print("\n")


--------HEAD DATA--------

    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


--------INFO DATA--------

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accident

In [3]:
#2. Clean target formatting
print("\n--------CLEAN--------\n")

df["Price"] = (df["Price"].replace(r"[\$,]", "", regex=True).astype(float))

print("\nPrice dtype:")
print(df["Price"].dtype)

print("\nPrice skewness:")
print(df["Price"].skew())

print("\n")


--------CLEAN--------


Price dtype:
float64

Price skewness:
7.871113660850301




In [4]:
#3. Fix Location
print("\n--------FIX LOCATION--------\n")

df["Location"] = df["Location"].replace({"Subrb": "Suburb","??": pd.NA})

print(df["Location"].value_counts(dropna=False))
print("\n")


--------FIX LOCATION--------

Location
City      59
Suburb    53
Rural     21
<NA>       7
NaN        5
Name: count, dtype: int64




In [5]:
#4. Fill missing values
print("\n--------Fill missing values--------\n")

df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

print("\nMissing after imputation:")
print(df.isnull().sum())
print("\n")


--------Fill missing values--------


Missing after imputation:
Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64




In [6]:
#5. Remove duplicates
print("\n--------Remove duplicates--------\n")

before = df.shape
df = df.drop_duplicates()
after = df.shape

print("\nBefore:", before)
print("After:", after)
print("Rows removed:", before[0] - after[0])

print("\n")


--------Remove duplicates--------


Before: (145, 6)
After: (140, 6)
Rows removed: 5




In [7]:
# 6. Handle Outliers (IQR Capping)

print("\n-------- Handle Outliers --------\n")

# Make sure columns are numeric
df["Price"] = pd.to_numeric(df["Price"])
df["Odometer_km"] = pd.to_numeric(df["Odometer_km"])

def iqr_bounds(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - (k * iqr)
    upper = q3 + (k * iqr)

    return lower, upper

# Calculate bounds
low_p, high_p = iqr_bounds(df["Price"])
low_o, high_o = iqr_bounds(df["Odometer_km"])

# Print bounds
print("Price Bounds:")
print(f"Lower Bound: {low_p}")
print(f"Upper Bound: {high_p}")

print("\nOdometer_km Bounds:")
print(f"Lower Bound: {low_o}")
print(f"Upper Bound: {high_o}")

# Apply IQR capping
df["Price"] = df["Price"].clip(lower=low_p, upper=high_p)
df["Odometer_km"] = df["Odometer_km"].clip(lower=low_o, upper=high_o)

# Summary after capping
print("\nSummary after capping:")
print(df[["Price", "Odometer_km"]].describe())

print("\nOutlier handling completed.\n")


-------- Handle Outliers --------

Price Bounds:
Lower Bound: -2984.625
Upper Bound: 8974.375

Odometer_km Bounds:
Lower Bound: -6642.25
Upper Bound: 271987.75

Summary after capping:
             Price    Odometer_km
count   140.000000     140.000000
mean   3175.456250  130945.403571
std    2601.848629   53815.006935
min    1500.000000    5000.000000
25%    1500.000000   97844.000000
50%    1500.000000  128548.000000
75%    4489.750000  167501.500000
max    8974.375000  271987.750000

Outlier handling completed.



In [8]:
#7. One-Hot Encoding
print("\n-------- One-Hot Encoding --------\n")

df = pd.get_dummies(df,columns=["Location"],drop_first=False,dtype="int")

print("\nNew columns:")
print(df.columns)

print("\n")


-------- One-Hot Encoding --------


New columns:
Index(['Price', 'Odometer_km', 'Doors', 'Accidents', 'Year', 'Location_City',
       'Location_Rural', 'Location_Suburb'],
      dtype='object')




In [9]:
#8. Create new features
print("\n--------Create new features--------\n")

CURRENT_YEAR = 2026

# Car age
df["CarAge"] = CURRENT_YEAR - df["Year"]
# Safe division
df["Km_per_year"] = (df["Odometer_km"] / df["CarAge"].replace(0, np.nan))
# Accident density
df["Accidents_per_Door"] = (df["Accidents"] / df["Doors"])
# Urban flag
df["Is_Urban"] = (df["Location_City"]).astype(int)
# Alternative target
df["LogPrice"] = np.log1p(df["Price"])

print(df.head())

print("\n")


--------Create new features--------

    Price  Odometer_km  Doors  Accidents  Year  Location_City  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998              1               0   
1  4171.0     128548.0    4.0          0  2016              0               1   
2  5331.0     107302.0    4.0          0  2014              0               0   
3  1500.0     141838.0    4.0          1  1999              0               0   
4  1500.0     128548.0    3.0          0  2022              1               0   

   Location_Suburb  CarAge   Km_per_year  Accidents_per_Door  Is_Urban  \
0                0      28   4922.500000                0.25         1   
1                0      10  12854.800000                0.00         0   
2                1      12   8941.833333                0.00         0   
3                1      27   5253.259259                0.25         0   
4                0       4  32137.000000                0.00         1   

   LogPrice  
0  7.313887  
1 

In [10]:
#9. Feature Scaling
print("\n--------Feature Scaling--------\n")

dont_scale = {"Price","LogPrice"}
numeric_cols = (df.select_dtypes(include=["int64", "float64"]).columns)

exclude = [
    c for c in df.columns
    if c.startswith("Location_")
]
exclude.append("Is_Urban")
to_scale = [
    c for c in numeric_cols
    if c not in dont_scale
    and c not in exclude
]
scaler = StandardScaler()
df[to_scale] = (scaler.fit_transform(df[to_scale]))

print("\nScaled sample:")
print(df[to_scale].head())

print("\n")


--------Feature Scaling--------


Scaled sample:
   Odometer_km     Doors  Accidents      Year    CarAge  Km_per_year  \
0     0.128390  0.254091   0.316968 -1.686714  1.686714    -0.615631   
1    -0.044709  0.254091  -0.820867  0.794617 -0.794617     0.070446   
2    -0.440923  0.254091  -0.820867  0.518913 -0.518913    -0.267993   
3     0.203135  0.254091   0.316968 -1.548862  1.548862    -0.587024   
4    -0.044709 -0.931668  -0.820867  1.621727 -1.621727     1.738196   

   Accidents_per_Door  
0            0.230749  
1           -0.815844  
2           -0.815844  
3            0.230749  
4           -0.815844  




In [11]:
#10. Final checks + Save
print("\n-------- Final checks + Save --------\n")

print("\nFINAL INFO")
print(df.info())

print("\nFINAL MISSING")
print(df.isnull().sum())

print("\nDESCRIBE")
print(df.describe())


# Assertions
assert df["Price"].dtype == float

assert "LogPrice" in df.columns

assert df.isnull().sum().sum() == 0

assert any(
    c.startswith("Location_")
    for c in df.columns
)

# Save
OUT = "./clean_car_dataset.csv"

df.to_csv(
    OUT,
    index=False
)

print("\nSaved:", OUT)

print("\n")


-------- Final checks + Save --------


FINAL INFO
<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 139
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Price               140 non-null    float64
 1   Odometer_km         140 non-null    float64
 2   Doors               140 non-null    float64
 3   Accidents           140 non-null    float64
 4   Year                140 non-null    float64
 5   Location_City       140 non-null    int64  
 6   Location_Rural      140 non-null    int64  
 7   Location_Suburb     140 non-null    int64  
 8   CarAge              140 non-null    float64
 9   Km_per_year         140 non-null    float64
 10  Accidents_per_Door  140 non-null    float64
 11  Is_Urban            140 non-null    int64  
 12  LogPrice            140 non-null    float64
dtypes: float64(9), int64(4)
memory usage: 15.3 KB
None

FINAL MISSING
Price                 0
Odometer_km       